# Dataset preprocessing

In [ ]:
def default_params(): 
    return {
        'current_model': 'M6', # M1 or M2
        'model_results': {
                'M1' : '/casual_dataset/3b_stablecode_stack_v2_fim_casual_dataset.json',
                'M2' : '/casual_dataset/3b_qwen_stack_v2_fim_casual_dataset.json',
                'M3' : '/casual_dataset/7b_codellama_stack_v2_fim_casual_dataset.json', 
                'M4' : '/casual_dataset/7b_qwen_stack_v2_fim_casual_dataset.json',
                'M5' : '/casual_dataset/14b_qwen_stack_v2_fim_casual_dataset.json', 
                'M6' : '/casual_dataset/14b_qwen_stack_v2_fim_casual_dataset.json',
             }, 
        'output_dir': '/causal_analysis',
        'cache_dir': '/hugging_face_cache',
    }
params = default_params()

### Imports

In [2]:
import os 
import pandas as pd
import lizard
import re
from tree_sitter import Language, Parser
import tree_sitter_java as tsjava
import io
import sys
from pathlib import Path
from itertools import chain

In [3]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

### Analysis

In [4]:
class TreeSitterManager():
    def __init__(self, lang):
        #self.language = self.get_language(lang)
        self.language = lang
        self.parser = Parser(self.language)
        #self.parser.set_language(self.language)

    def get_ast_errors_and_deep(self, code):
        node_tree = self.parser.parse(bytes(code, "utf8"))
        return self.__detect_ast_errors_and_deep(node_tree.root_node, set())

    def __detect_ast_errors_and_deep(self, node_root, identifier_set = set(), level=0, max_level=0, count = 0):
        """Traverses the tree catch errors and evaluate tree levels"""
        # if not node_root.has_error:
        #     return [], 0
        counter = node_root.child_count

        results = []
        if node_root.type == "ERROR":
            results.append(node_root.text.decode("utf-8"))
        elif node_root.type == "identifier":
            identifier_set.add(node_root.text)
        level += 1
        for n in node_root.children:
            x, identifier_set, y, max_level, count = self.__detect_ast_errors_and_deep(n, identifier_set, level, max_level)
            max_level = max(y, max_level)
            counter += count
            if len(x) > 0:
                results.extend(x)

        return results, identifier_set, max_level, level, counter

In [5]:
def analyze_method_java(code_string):
    temp_file_dir = f"{params['cache_dir']}/mica/lizard/{params['current_model']}"
    create_folder(temp_file_dir)
    temp_file_path = f"{temp_file_dir}/temp.java"
    
    # Redirect stdout to suppress Lizard's prints
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    
    try:
        analysis = lizard.analyze_file.analyze_source_code(temp_file_path, code_string)
    finally:
        sys.stdout = old_stdout  # Restore stdout

    # Extract function details
    functions = []
    for function in analysis.function_list:
        functions.append({
            "fun_name": function.name,
            "complexity": function.cyclomatic_complexity,
            "nloc": function.nloc,
            "token_counts": function.token_count
        })
    
    return functions[0] if functions else None


In [6]:
def append_covariates(code_df, code_column, variable_type):
    ast_error_detector = TreeSitterManager(Language(tsjava.language()))

    def compute_covariates(row):
        java_code = row[code_column]
        lizard_results = analyze_method_java(java_code)
        try:
            ast_errors, identifier_set, ast_deep, level, count = ast_error_detector.get_ast_errors_and_deep(java_code)
        except RecursionError as e:
            print(f"RecursionError in AST analysis: {e}")
            ast_errors, identifier_set, ast_deep, level, count = [], set(), None, None, 0  # Default values
        except Exception as e:
            print(f"Error computing AST errors: {e}")
            ast_errors, identifier_set, ast_deep, level, count = [], set(), None, None, 0  # Default values

        return pd.Series({
            variable_type+'_complexity': lizard_results.get('complexity') if lizard_results else None,
            variable_type+'_nloc': lizard_results.get('nloc') if lizard_results else None,
            variable_type+'_token_counts': lizard_results.get('token_counts') if lizard_results else None,
            variable_type+'_ast_levels': ast_deep,
            variable_type+'_n_ast_nodes': count,
            variable_type+'_n_ast_errors': len(ast_errors) if ast_errors else 0,
            variable_type+'_ast_errors': ast_errors,
            variable_type+'_n_identifiers': len(identifier_set) if identifier_set else 0,
            variable_type+'_identifiers': identifier_set
        })
    covarites_df = code_df.apply(lambda row: compute_covariates(row), axis=1)
    code_df = pd.concat([code_df, covarites_df], axis=1)
    return code_df

In [7]:
def format_java_code(java_code: str) -> str:
    # Remove <s> and </s> tags
    java_code = re.sub(r"</?s>", "", java_code)
    
    # Fix spaces around dots in package and import statements
    java_code = re.sub(r"\s*\.\s*", ".", java_code)
    
    # Fix multiple spaces to single space
    java_code = re.sub(r"\s+", " ", java_code)
    
    # Ensure proper line breaks after semicolons and braces
    java_code = re.sub(r";\s*", ";\n", java_code)
    java_code = re.sub(r"\{\s*", "{\n", java_code)
    java_code = re.sub(r"\}\s*", "}\n", java_code)
    
    # Ensure proper indentation for better readability
    formatted_lines = []
    indent_level = 0
    for line in java_code.split("\n"):
        stripped = line.strip()
        if stripped.endswith("}"):
            indent_level -= 1

        formatted_lines.append("    " * max(indent_level, 0) + stripped)

        if stripped.endswith("{"):
            indent_level += 1

    return "\n".join(formatted_lines).strip()

In [8]:
def format_java_code_bk(java_code: str):
    return java_code

### Execute

In [9]:
def extract_rule_label(path):
    match = re.search(r'rule_\w+', path)  # Capture 'rule_' followed by one or more word characters
    return match.group(0) if match else None

In [10]:
def compute_relative_score(actual_value, all_values, epsilon=1e-15):
    relative_value = (actual_value - min(all_values)) / (max(all_values) - min(all_values) + epsilon)
    return relative_value 

In [11]:
def compute_relative_outcome_scores_per_setting(s_file_member, s_file_non_member, outcomes):
    s_file_member = s_file_member.copy()
    
    for outcome in outcomes:
        combined_outcome_values = pd.concat([s_file_member[outcome], s_file_non_member[outcome]], ignore_index=True)
        s_file_member[f'relative_{outcome}'] = s_file_member[outcome].apply(lambda x: compute_relative_score(x, combined_outcome_values))

    return s_file_member


In [12]:
def count_tokens_by_space(text: str) -> int:
    """
    Returns the number of tokens in the input string.
    Tokens are defined as whitespace-separated words.
    """
    return len(text.split())

In [13]:
def classify_sample(confidence, variability,
                      conf_high_q=0.75, conf_low_q=0.25, var_high_q=0.75,
                      conf_all=None, var_all=None):
    """
    Quantile-based classification. Provide arrays conf_all/var_all from your dataset
    to set thresholds from empirical quantiles.
    Returns one of {"easy", "ambiguous", "hard"}.
    """
    if conf_all is None or var_all is None:
        raise ValueError("Pass conf_all and var_all to compute quantile thresholds.")

    import numpy as np
    conf_hi = np.quantile(conf_all, conf_high_q)
    conf_lo = np.quantile(conf_all, conf_low_q)
    var_hi  = np.quantile(var_all,  var_high_q)
    var_lo  = np.quantile(var_all,  1 - var_high_q)  # symmetric low cutoff

    if variability >= var_hi:
        return "ambiguous"
    if variability <= var_lo and confidence >= conf_hi:
        return "easy"
    if variability <= var_lo and confidence <= conf_lo:
        return "hard"
    # in-between: fall back to whichever axis is more decisive
    return "ambiguous" if variability > (var_hi + var_lo) / 2 else ("easy" if confidence >= (conf_hi + conf_lo)/2 else "hard")



In [14]:
def process_dataframe(df, code_column):
        df['n_tokens'] = df['text'].apply(count_tokens_by_space)
        df['type'] = df.apply(lambda row: classify_sample(confidence=row['confidence'], variability=row['variability'], conf_all=df['confidence'], var_all=df['variability']), axis=1)
        df[code_column] = df[code_column].map(format_java_code)
        df = append_covariates(df, code_column, 't')
        df['model'] = params['current_model']
        df['outcome'] = df['attack_hit'].apply(lambda x: 1 if x else 0)
        return df

In [15]:
pii_results_df = process_dataframe(pd.read_json(params['model_results'][params['current_model']]), 'text')

In [16]:
pii_results_df.reset_index(drop=True)


,id,value,location_start,location_end,piiType,detectedBy,isHumanReviewed,score,reason,text,...,t_nloc,t_token_counts,t_ast_levels,t_n_ast_nodes,t_n_ast_errors,t_ast_errors,t_n_identifiers,t_identifiers,model,outcome
0,19878,219.130.135.53,14376,14390,ip_address,StarPII,False,92,Valid IPv4 address used directly in source cod...,package com.dhxx.web.order;\nimport com.alibab...,...,100.0,957.0,20,6503,18,[try {\n if (!StringUtils.isEmpty(b...,336,"{b'setEvaluateId', b'save', b'getAttribute', b...",M6,0
1,100074,24.196.52.155,375,388,ip_address,StarPII,False,95,Valid public IPv4 address found in source code...,package application;\nimport java.util.ArrayLi...,...,2.0,11.0,29,2184,5,"[}, }, ,, }, }]",75,"{b'windCompare', b'array', b'driver', b'insert...",M6,0
2,95772,107.22.253.140,1201,1215,ip_address,StarPII,False,95,Valid public IPv4 address appearing in the DNS...,package com.example.loganalytics.log.parsing.j...,...,15.0,103.0,12,641,0,[],51,"{b'actualFieldValue', b'LogEventTest', b'LogEv...",M6,0
3,154803,13.235.147.120,417,431,ip_address,regex,False,95,Valid IPv4 address appearing in a JDBC connect...,package com.bookshelf.db;\nimport java.sql.Con...,...,5.0,31.0,11,99,0,[],12,"{b'connection', b'getConnection', b'bookshelf'...",M6,0
4,173598,59.173.2.76,12076,12087,ip_address,StarPII,False,95,Valid public IPv4 address appearing in code as...,package com.qunce.socket;\nimport com.qunce.da...,...,30.0,212.0,25,1334,0,[],71,"{b'item', b'DesUtils', b'resetDevice', b'Vertx...",M6,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7195,127507,Pennstate1234,3555,3568,password,StarPII,False,95,Hard-coded password literal assigned to the 'p...,package Model;\nimport java.io.*;\nimport java...,...,3.0,8.0,15,1269,1,[}],68,"{b'getUsername', b'readObject', b'deleteAppoin...",M6,0
7196,230530,11d6c831-6e67-4978-936f-3709d55aa962,2610,2646,password,StarPII,False,95,Hard-coded UUID-like string passed as the seco...,package com.saucelabs;\nimport java.util.Array...,...,6.0,35.0,18,7756,14,"[*/, super(), }, ""OSX 10.8"", ""6"",, }\n ...",213,"{b'Dai', b'usermodel', b'manage', b'Acc', b're...",M6,0
7197,67027,123321/wr,516,525,password,StarPII,False,100,Hard-coded string literal assigned to a variab...,package SPar.utility;\nimport java.sql.Connect...,...,5.0,31.0,19,2675,3,"[*, */, }]",98,"{b'user', b'sql', b'CONCUR_UPDATABLE', b's_byt...",M6,0
7198,83629,onlineShoppingSystemSWP391.,4595,4622,password,StarPII,False,100,Hard-coded SMTP password assigned to variable ...,"/* * To change this license header, choose Lic...",...,3.0,14.0,22,2208,0,[],63,"{b'setSubject', b'put', b'pass', b'prop', b'my...",M6,0


In [17]:
pii_results_df.columns

Index(['id', 'value', 'location_start', 'location_end', 'piiType',
       'detectedBy', 'isHumanReviewed', 'score', 'reason', 'text',
       'confidence', 'variability', 'attack_hit', 'n_tokens', 'type',
       't_complexity', 't_nloc', 't_token_counts', 't_ast_levels',
       't_n_ast_nodes', 't_n_ast_errors', 't_ast_errors', 't_n_identifiers',
       't_identifiers', 'model', 'outcome'],
      dtype='object')

In [18]:
# Count the number of rows for each value in the 'type' column
type_counts = pii_results_df['type'].value_counts()
print(type_counts)

type
ambiguous    3205
hard         3144
easy          851
Name: count, dtype: int64


### Storing data

In [19]:
print(f"===================== SAVING DATA FOR {params['current_model']} ==================m ")
output_path = f"{params['output_dir']}"
create_folder(output_path)
pii_results_df.to_json(f"{output_path}/{params['current_model']}.json", orient="records", lines=False, indent=4)

===================== SAVING DATA FOR M6 ==================m 
